<a href="https://colab.research.google.com/github/ParisaAligol/CityLearn-EVModel-Parisa/blob/master/MPC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import Any, List
import numpy as np
from scipy.optimize import minimize
from citylearn.citylearn import CityLearnEnv
from citylearn.preprocessing import Encoder, PeriodicNormalization, Normalize, OnehotEncoding

# conditional imports
try:
    import torch
except (ModuleNotFoundError, ImportError) as e:
    raise Exception("This functionality requires you to install torch. You can install torch by : pip install torch torchvision, or for more detailed instructions please visit https://pytorch.org.")

from citylearn.agents.base import Agent

class MPC(Agent):

    def __init__(
        self, env: CityLearnEnv, hidden_dimension: List[float] = None,
        # MPC Parameters
        horizon: int = 6,
        comfort_temp: float = 22.0,
        comfort_range: Tuple[float, float] = (20.0, 24.0),
        alpha: float = 1.0,                # comfort weight
        beta: float = 2.0,                 # energy weight
        # LSTM features
        lstm_model: Optional[torch.nn.Module] = None,
        num_features: int = 13,
        seq_len: int = 24,
        lstm_device: Optional[str] = None,
        # control mapping
        control_building_index: int = 0,   # which building we control with setpoint
        setpoint_action_index: int = 0,    # index in the action vector to write setpoint
        # optimization
        maxiter: int = 500,
        ftol: float = 1e-3,
        **kwargs: Any,
    ):
        """MPC Agent that solves a small SLSQP each step and returns the first action."""
        super().__init__(env, **kwargs)

        self.h = horizon
        self.comfort_temp = comfort_temp
        self.comfort_low, self.comfort_high = comfort_range
        self.alpha = alpha
        self.beta = beta

        self.num_features = int(num_features)
        self.seq_len = int(seq_len)
        self.control_bldg = int(control_building_index)
        self.setpoint_action_index = int(setpoint_action_index)

        # Torch/LSTM
        self.device = torch.device(
            lstm_device if lstm_device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        if lstm_model is None:
            raise ValueError("You must pass a trained `lstm_model` to MPC(...)")
        self.lstm = lstm_model.to(self.device).eval()

        # SLSQP options
        self._slsqp_opts = dict(maxiter=maxiter, ftol=ftol, disp=False)

        # normalization flags (reuse RLC helpers, like your SAC does)
        self.normalized = [False for _ in self.action_space]
        self.norm_mean = [None for _ in self.action_space]
        self.norm_std = [None for _ in self.action_space]

        # encoders (e.g., remove features you don’t need)
        self.encoders = self.set_encoders()


    def predict(self, observations: List[List[float]], deterministic: bool = None) -> List[List[float]]:
        """Solve MPC (receding horizon) and return first-step action(s)."""
        # Build default actions (zeros), then overwrite the controlled entry
        actions: List[List[float]] = [list(np.zeros_like(a.low)) for a in self.action_space]

        # Prepare inputs for the controlled building only
        b = self.control_bldg
        obs_b = self._encode_obs(b, observations[b])

        # Current indoor/outdoor temps (indices resolved once on first call)
        if not hasattr(self, "_idx"):
            self._cache_indices()

        indoor = obs_b[self._idx["indoor"]]
        outdoor = obs_b[self._idx["outdoor"]]

        # Build exogenous inputs for horizon (repeat last known values or pull forecasts if you have them)
        external_inputs = [np.array([indoor, outdoor], dtype=float) for _ in range(self.h)]

        # Solve small SLSQP for setpoints over horizon
        u_star = self._solve_mpc(initial_state=np.array([indoor], dtype=float), external_inputs=external_inputs)

        # Recede: first action only
        setpoint = float(u_star[0])

        # Write setpoint into the action vector for building b
        a_vec = np.array(actions[b], dtype=float)
        a_vec[self.setpoint_action_index] = setpoint
        actions[b] = a_vec.tolist()

        # advance agent time step like in SAC
        self.actions = actions
        self.next_time_step()
        return actions


    def _cache_indices(self):
        """Resolve observation indices once, using env.observation_names like your Colab code."""
        column_names = self.observation_names[self.control_bldg]
        def idx(name: str) -> int:
            try:
                return column_names.index(name)
            except ValueError as e:
                raise ValueError(f"Observation '{name}' not found in env.observation_names[{self.control_bldg}]") from e
        self._idx = {
            "indoor": idx("indoor_dry_bulb_temperature"),
            "outdoor": idx("outdoor_dry_bulb_temperature"),
            # Add more if your LSTM uses them
            # "setpoint": idx("indoor_dry_bulb_temperature_cooling_set_point"),
            # "electricity": idx("net_electricity_consumption"),
            # "solar": idx("solar_generation"),
        }

    def _encode_obs(self, index: int, observations: List[float]) -> npt.NDArray[np.float64]:
        # Mirror your SAC: encoders then optional normalization
        o = self.get_encoded_observations(index, observations)
        if self.normalized[index] and (self.norm_mean[index] is not None):
            o = (o - self.norm_mean[index]) / (self.norm_std[index] + 1e-8)
        return o

    def _solve_mpc(self, initial_state: npt.NDArray[np.float64], external_inputs: List[np.ndarray]) -> np.ndarray:
        # bounds & initial guess
        u0 = [self.comfort_temp] * self.h
        bounds = [(self.comfort_low, self.comfort_high) for _ in range(self.h)]

        def cost(u_flat: np.ndarray) -> float:
            return self._mpc_cost_function(u_flat, initial_state, external_inputs)

        res = minimize(cost, x0=u0, bounds=bounds, method="SLSQP", options=self._slsqp_opts)
        if not res.success:
            # Fall back to feasible constant setpoint if optimize fails
            return np.array(u0, dtype=float)
        return res.x

    def _mpc_cost_function(
        self,
        control_sequence: np.ndarray,
        initial_state: npt.NDArray[np.float64],
        external_inputs: List[np.ndarray],
    ) -> float:
        """Your Colab cost, but scoped to this agent and with shapes handled safely."""
        total_cost = 0.0

        # Prepare LSTM hidden state (batch=1)
        batch_size = 1
        device = self.device

        # You must provide init_hidden(here) in your LSTM module; else adapt accordingly
        hidden = self.lstm.init_hidden(batch_size, device)

        current_state = np.array(initial_state, dtype=float).reshape(-1)

        for t, u in enumerate(control_sequence):
            # Build feature vector: [state pad..., external..., control]
            ext = np.array(external_inputs[min(t, len(external_inputs)-1)], dtype=float).reshape(-1)
            control = np.array([u], dtype=float)

            # Pad state so that [state, ext, control] == num_features
            need = self.num_features - (current_state.size + ext.size + control.size)
            if need < 0:
                raise ValueError(f"num_features={self.num_features} too small for state+ext+control at step {t}")
            state_padded = np.pad(current_state, (0, need), mode="constant")

            x = np.concatenate([state_padded, ext, control]).astype(np.float32)
            x = torch.from_numpy(x).reshape(1, 1, -1).to(device)

            # LSTM forward (expects (B, T, F) or (T, B, F) per your implementation)
            with torch.no_grad():
                y_pred, hidden = self.lstm(x, hidden)  # y_pred should be shape (1, 1, 1) for temperature
            predicted_temp = float(y_pred.item())

            # Comfort penalty (outside band)
            high_violation = max(0.0, predicted_temp - self.comfort_high)
            low_violation = max(0.0, self.comfort_low - predicted_temp)
            comfort_cost = self.alpha * (high_violation**2 + low_violation**2)

            # Energy proxy: scale by absolute control move; replace with a proper grid import model if available
            # Using building net consumption at current time step if exposed; else use a proxy weight.
            try:
                price_signal = 1.0
                energy_cost = self.beta * price_signal * abs(control[0])
            except Exception:
                energy_cost = self.beta * abs(control[0])

            total_cost += (comfort_cost + energy_cost)
            current_state = np.array([predicted_temp], dtype=float)

        return float(total_cost)

    def set_encoders(self) -> List[List[Encoder]]:
        encoders = super().set_encoders()
        for i, names in enumerate(self.observation_names):
            for j, n in enumerate(names):
                # Example: remove net_electricity_consumption as SAC did
                if n == "net_electricity_consumption":
                    encoders[i][j] = RemoveFeature()
        return encoders